# Survived (Titanic) Life Prediction with Apache Spark

このNotebookでは、Apache Sparkを使用してSurvivedデータセット（タイタニック号の生存者データ）を探索し、生存予測の分類モデルを構築します。

In [2]:
// Spark 依存関係の読み込み（Scala 2.13 を明示的に指定）
import $ivy.`org.apache.spark:spark-sql_2.13:3.5.0`
import $ivy.`org.apache.spark:spark-mllib_2.13:3.5.0`

println("Spark 依存関係が正常にロードされました")

Spark 依存関係が正常にロードされました


import $ivy.$
import $ivy.$

## 1. 環境設定とライブラリのインポート

In [3]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.ml.{Pipeline, PipelineModel}
import org.apache.spark.ml.classification.LogisticRegression
import org.apache.spark.ml.feature.{Imputer, StringIndexer, OneHotEncoder, VectorAssembler}
import org.apache.spark.ml.evaluation.{BinaryClassificationEvaluator, MulticlassClassificationEvaluator}

// SparkSessionの作成
val spark = SparkSession.builder()
  .appName("SurvivedExploration")
  .master("local[*]")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

println("Spark Session created successfully!")
println(s"Spark version: ${spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/04 13:22:15 INFO SparkContext: Running Spark version 3.5.0
25/11/04 13:22:15 INFO SparkContext: OS info Windows 11, 10.0, amd64
25/11/04 13:22:15 INFO SparkContext: Java version 21.0.2
25/11/04 13:22:15 WARN Shell: Did not find winutils.exe: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://wiki.apache.org/hadoop/WindowsProblems
25/11/04 13:22:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/04 13:22:16 INFO ResourceUtils: ==============================================================
25/11/04 13:22:16 INFO ResourceUtils: No custom resources configured for spark.driver.
25/11/04 13:22:16 INFO ResourceUtils: ==============================================================
25/11/04 13:22:16 INFO SparkContext: Submitted application: SurvivedExploration
25

Spark Session created successfully!
Spark version: 3.5.0


import org.apache.spark.sql.SparkSession
import org.apache.spark.ml.{Pipeline, PipelineModel}
import org.apache.spark.ml.classification.LogisticRegression
import org.apache.spark.ml.feature.{Imputer, StringIndexer, OneHotEncoder, VectorAssembler}
import org.apache.spark.ml.evaluation.{BinaryClassificationEvaluator, MulticlassClassificationEvaluator}
spark: SparkSession = org.apache.spark.sql.SparkSession@d7df245

## 2. データの読み込み

In [4]:
// データの読み込み
val df = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("../data/Survived.csv")

// カラム名を小文字に変換
val lowercaseDF = df.columns.foldLeft(df) { (currentDF, colName) =>
  currentDF.withColumnRenamed(colName, colName.toLowerCase)
}

println(s"データ件数: ${lowercaseDF.count()}")
println("\nスキーマ:")
lowercaseDF.printSchema()

データ件数: 891

スキーマ:
root
 |-- passengerid: integer (nullable = true)
 |-- survived: integer (nullable = true)
 |-- pclass: integer (nullable = true)
 |-- sex: string (nullable = true)
 |-- age: double (nullable = true)
 |-- sibsp: integer (nullable = true)
 |-- parch: integer (nullable = true)
 |-- ticket: string (nullable = true)
 |-- fare: double (nullable = true)
 |-- cabin: string (nullable = true)
 |-- embarked: string (nullable = true)



df: org.apache.spark.sql.package.DataFrame = [PassengerId: int, Survived: int ... 9 more fields]
lowercaseDF: org.apache.spark.sql.package.DataFrame = [passengerid: int, survived: int ... 9 more fields]

## 3. データの概要確認

In [5]:
// 最初の10行を表示
lowercaseDF.show(10, truncate = false)

+-----------+--------+------+------+----+-----+-----+----------------+-------+-----+--------+
|passengerid|survived|pclass|sex   |age |sibsp|parch|ticket          |fare   |cabin|embarked|
+-----------+--------+------+------+----+-----+-----+----------------+-------+-----+--------+
|1          |0       |3     |male  |22.0|1    |0    |A/5 21171       |7.25   |NULL |S       |
|2          |1       |1     |female|38.0|1    |0    |PC 17599        |71.2833|C85  |C       |
|3          |1       |3     |female|26.0|0    |0    |STON/O2. 3101282|7.925  |NULL |S       |
|4          |1       |1     |female|35.0|1    |0    |113803          |53.1   |C123 |S       |
|5          |0       |3     |male  |35.0|0    |0    |373450          |8.05   |NULL |S       |
|6          |0       |3     |male  |NULL|0    |0    |330877          |8.4583 |NULL |Q       |
|7          |0       |1     |male  |54.0|0    |0    |17463           |51.8625|E46  |S       |
|8          |0       |3     |male  |2.0 |3    |1    |349909 

In [6]:
// 統計情報
lowercaseDF.describe("age", "sibsp", "parch", "fare").show()

+-------+------------------+------------------+-------------------+-----------------+
|summary|               age|             sibsp|              parch|             fare|
+-------+------------------+------------------+-------------------+-----------------+
|  count|               714|               891|                891|              891|
|   mean| 29.69911764705882|0.5230078563411896|0.38159371492704824| 32.2042079685746|
| stddev|14.526497332334035|1.1027434322934315| 0.8060572211299488|49.69342859718089|
|    min|              0.42|                 0|                  0|              0.0|
|    max|              80.0|                 8|                  6|         512.3292|
+-------+------------------+------------------+-------------------+-----------------+



In [7]:
// 生存者数の確認
lowercaseDF.groupBy("survived").count().show()

+--------+-----+
|survived|count|
+--------+-----+
|       1|  342|
|       0|  549|
+--------+-----+



In [8]:
// 性別ごとの生存率
lowercaseDF.groupBy("sex", "survived").count().show()

+------+--------+-----+
|   sex|survived|count|
+------+--------+-----+
|  male|       0|  468|
|female|       1|  233|
|female|       0|   81|
|  male|       1|  109|
+------+--------+-----+



In [9]:
// クラスごとの生存率
lowercaseDF.groupBy("pclass", "survived").count().orderBy("pclass").show()

+------+--------+-----+
|pclass|survived|count|
+------+--------+-----+
|     1|       0|   80|
|     1|       1|  136|
|     2|       1|   87|
|     2|       0|   97|
|     3|       1|  119|
|     3|       0|  372|
+------+--------+-----+



## 4. 欠損値の確認と補完

In [10]:
// 欠損値の確認
println("欠損値の確認:")
lowercaseDF.select("age", "fare").summary("count").show()

// 欠損値補完（平均値で補完）
val imputer = new Imputer()
  .setInputCols(Array("age", "fare"))
  .setOutputCols(Array("age_imputed", "fare_imputed"))
  .setStrategy("mean")

val imputedDF = imputer.fit(lowercaseDF).transform(lowercaseDF)

println("\n欠損値補完完了")
imputedDF.select("age", "age_imputed", "fare", "fare_imputed").show(5)

欠損値の確認:
+-------+---+----+
|summary|age|fare|
+-------+---+----+
|  count|714| 891|
+-------+---+----+


欠損値補完完了
+----+-----------+-------+------------+
| age|age_imputed|   fare|fare_imputed|
+----+-----------+-------+------------+
|22.0|       22.0|   7.25|        7.25|
|38.0|       38.0|71.2833|     71.2833|
|26.0|       26.0|  7.925|       7.925|
|35.0|       35.0|   53.1|        53.1|
|35.0|       35.0|   8.05|        8.05|
+----+-----------+-------+------------+
only showing top 5 rows



imputer: Imputer = imputer_893b083bb26c
imputedDF: org.apache.spark.sql.package.DataFrame = [passengerid: int, survived: int ... 11 more fields]

## 5. カテゴリカル変数のエンコーディング

In [11]:
// sex（性別）のエンコーディング
val sexIndexer = new StringIndexer()
  .setInputCol("sex")
  .setOutputCol("sex_index")

val sexEncoder = new OneHotEncoder()
  .setInputCol("sex_index")
  .setOutputCol("sex_vec")

// embarked（乗船港）のエンコーディング
val embarkedIndexer = new StringIndexer()
  .setInputCol("embarked")
  .setOutputCol("embarked_index")
  .setHandleInvalid("keep")

val embarkedEncoder = new OneHotEncoder()
  .setInputCol("embarked_index")
  .setOutputCol("embarked_vec")

val encodePipeline = new Pipeline().setStages(Array(
  sexIndexer, sexEncoder,
  embarkedIndexer, embarkedEncoder
))

val encodedDF = encodePipeline.fit(imputedDF).transform(imputedDF)

println("カテゴリカル変数のエンコーディング完了")
encodedDF.select("sex", "sex_vec", "embarked", "embarked_vec").show(5, truncate = false)

カテゴリカル変数のエンコーディング完了
+------+-------------+--------+-------------+
|sex   |sex_vec      |embarked|embarked_vec |
+------+-------------+--------+-------------+
|male  |(1,[0],[1.0])|S       |(3,[0],[1.0])|
|female|(1,[],[])    |C       |(3,[1],[1.0])|
|female|(1,[],[])    |S       |(3,[0],[1.0])|
|female|(1,[],[])    |S       |(3,[0],[1.0])|
|male  |(1,[0],[1.0])|S       |(3,[0],[1.0])|
+------+-------------+--------+-------------+
only showing top 5 rows



sexIndexer: StringIndexer = strIdx_ec84d9fca6a5
sexEncoder: OneHotEncoder = oneHotEncoder_d188ddc14380
embarkedIndexer: StringIndexer = strIdx_6b4b438cf8c3
embarkedEncoder: OneHotEncoder = oneHotEncoder_de2f9fd0d545
encodePipeline: Pipeline = pipeline_5ccfcdfc3e07
encodedDF: org.apache.spark.sql.package.DataFrame = [passengerid: int, survived: int ... 15 more fields]

## 6. 外れ値の除去

In [12]:
// 運賃の外れ値を確認
println("運賃の分布:")
encodedDF.select("fare_imputed").describe().show()

// 異常に高い運賃のデータを除去
val cleanedDF = encodedDF.filter("fare_imputed < 500 AND fare_imputed > 0")

println(s"\n外れ値除去前: ${encodedDF.count()} 件")
println(s"外れ値除去後: ${cleanedDF.count()} 件")
println(s"除去された件数: ${encodedDF.count() - cleanedDF.count()} 件")

運賃の分布:
+-------+-----------------+
|summary|     fare_imputed|
+-------+-----------------+
|  count|              891|
|   mean| 32.2042079685746|
| stddev|49.69342859718089|
|    min|              0.0|
|    max|         512.3292|
+-------+-----------------+


外れ値除去前: 891 件
外れ値除去後: 873 件
除去された件数: 18 件


cleanedDF: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [passengerid: int, survived: int ... 15 more fields]

## 7. 特徴量の統合

In [13]:
// 全ての特徴量を1つのベクトルに統合
val assembler = new VectorAssembler()
  .setInputCols(Array(
    "pclass",
    "age_imputed",
    "sibsp",
    "parch",
    "fare_imputed",
    "sex_vec",
    "embarked_vec"
  ))
  .setOutputCol("features")

val assembledDF = assembler.transform(cleanedDF)

println(s"準備後のデータ件数: ${assembledDF.count()}")
assembledDF.select("features", "survived").show(5, truncate = false)

準備後のデータ件数: 873
+------------------------------------------+--------+
|features                                  |survived|
+------------------------------------------+--------+
|[3.0,22.0,1.0,0.0,7.25,1.0,1.0,0.0,0.0]   |0       |
|[1.0,38.0,1.0,0.0,71.2833,0.0,0.0,1.0,0.0]|1       |
|(9,[0,1,4,6],[3.0,26.0,7.925,1.0])        |1       |
|[1.0,35.0,1.0,0.0,53.1,0.0,1.0,0.0,0.0]   |1       |
|[3.0,35.0,0.0,0.0,8.05,1.0,1.0,0.0,0.0]   |0       |
+------------------------------------------+--------+
only showing top 5 rows



assembler: VectorAssembler = VectorAssembler: uid=vecAssembler_9c3fb639e54b, handleInvalid=error, numInputCols=7
assembledDF: org.apache.spark.sql.package.DataFrame = [passengerid: int, survived: int ... 16 more fields]

## 8. データの分割

In [14]:
// 訓練データとテストデータに分割
val Array(trainData, testData) = assembledDF.randomSplit(Array(0.7, 0.3), seed = 42)

println(s"訓練データ: ${trainData.count()} 件")
println(s"テストデータ: ${testData.count()} 件")

// クラスバランスの確認
println("\n訓練データのクラスバランス:")
trainData.groupBy("survived").count().show()

訓練データ: 648 件
テストデータ: 225 件

訓練データのクラスバランス:
+--------+-----+
|survived|count|
+--------+-----+
|       1|  260|
|       0|  388|
+--------+-----+



trainData: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [passengerid: int, survived: int ... 16 more fields]
testData: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [passengerid: int, survived: int ... 16 more fields]

## 9. Logistic Regressionモデルの訓練

In [15]:
// ラベルカラムをリネーム
val labeledTrain = trainData.withColumnRenamed("survived", "label")

// Logistic Regressionモデルの作成
val lr = new LogisticRegression()
  .setLabelCol("label")
  .setFeaturesCol("features")
  .setMaxIter(100)
  .setRegParam(0.01)

val pipeline = new Pipeline().setStages(Array(lr))

// モデルの訓練
println("モデルを訓練中...")
val model = pipeline.fit(labeledTrain)
println("訓練完了！")

モデルを訓練中...
訓練完了！


labeledTrain: org.apache.spark.sql.package.DataFrame = [passengerid: int, label: int ... 16 more fields]
lr: LogisticRegression = logreg_db27a6712be9
pipeline: Pipeline = pipeline_c646d2c5e53e
model: PipelineModel = pipeline_c646d2c5e53e

## 10. モデルの評価

In [16]:
// テストデータで予測
val labeledTest = testData.withColumnRenamed("survived", "label")
val predictions = model.transform(labeledTest)

// Accuracy（精度）
val correct = predictions.filter("prediction = label").count()
val total = predictions.count()
val accuracy = correct.toDouble / total

println(f"Accuracy: ${accuracy * 100}%.2f%%")

// AUC（Area Under ROC Curve）
val aucEvaluator = new BinaryClassificationEvaluator()
  .setLabelCol("label")
  .setRawPredictionCol("rawPrediction")
  .setMetricName("areaUnderROC")

val auc = aucEvaluator.evaluate(predictions)
println(f"AUC: ${auc * 100}%.2f%%")

// Precision（適合率）
val truePositives = predictions.filter("prediction = 1 AND label = 1").count().toDouble
val predictedPositives = predictions.filter("prediction = 1").count().toDouble
val precision = if (predictedPositives > 0) truePositives / predictedPositives else 0.0
println(f"Precision: ${precision * 100}%.2f%%")

// Recall（再現率）
val actualPositives = predictions.filter("label = 1").count().toDouble
val recall = if (actualPositives > 0) truePositives / actualPositives else 0.0
println(f"Recall: ${recall * 100}%.2f%%")

// F1 Score
val f1 = if (precision + recall > 0) 2 * (precision * recall) / (precision + recall) else 0.0
println(f"F1 Score: ${f1 * 100}%.2f%%")

Accuracy: 75.56%
AUC: 82.47%
Precision: 63.22%
Recall: 70.51%
F1 Score: 66.67%


labeledTest: org.apache.spark.sql.package.DataFrame = [passengerid: int, label: int ... 16 more fields]
predictions: org.apache.spark.sql.package.DataFrame = [passengerid: int, label: int ... 19 more fields]
correct: Long = 170L
total: Long = 225L
accuracy: Double = 0.7555555555555555
aucEvaluator: BinaryClassificationEvaluator = BinaryClassificationEvaluator: uid=binEval_1b957407b5d8, metricName=areaUnderROC, numBins=1000
auc: Double = 0.8246555032269314
truePositives: Double = 55.0
predictedPositives: Double = 87.0
precision: Double = 0.632183908045977
actualPositives: Double = 78.0
recall: Double = 0.7051282051282052
f1: Double = 0.6666666666666666

## 11. 予測結果の確認

In [17]:
// 予測結果のサンプル表示
predictions.select(
  "pclass", "sex", "age_imputed", "fare_imputed",
  "label", "prediction", "probability"
).show(15, truncate = false)

+------+------+-----------------+------------+-----+----------+----------------------------------------+
|pclass|sex   |age_imputed      |fare_imputed|label|prediction|probability                             |
+------+------+-----------------+------------+-----+----------+----------------------------------------+
|3     |female|26.0             |7.925       |1    |1.0       |[0.35827121145389623,0.6417287885461038]|
|1     |male  |54.0             |51.8625     |0    |0.0       |[0.6964138382302126,0.30358616176978737]|
|3     |female|27.0             |11.1333     |1    |1.0       |[0.38478516583926836,0.6152148341607316]|
|2     |female|14.0             |30.0708     |1    |1.0       |[0.1087503245975314,0.8912496754024686] |
|3     |male  |39.0             |31.275      |0    |0.0       |[0.9521504158617067,0.04784958413829332]|
|3     |female|14.0             |7.8542      |0    |1.0       |[0.2528161485560078,0.7471838514439921] |
|2     |female|55.0             |16.0        |1    |1.0

In [18]:
// 混同行列（Confusion Matrix）
println("混同行列:")
predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

混同行列:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0|  115|
|    0|       1.0|   32|
|    1|       0.0|   23|
|    1|       1.0|   55|
+-----+----------+-----+



In [19]:
// 誤分類されたケースの確認
println("誤分類されたケース（最初の10件）:")
predictions.filter("prediction != label")
  .select("pclass", "sex", "age_imputed", "sibsp", "parch", "fare_imputed", "label", "prediction")
  .show(10, truncate = false)

誤分類されたケース（最初の10件）:
+------+------+-----------------+-----+-----+------------+-----+----------+
|pclass|sex   |age_imputed      |sibsp|parch|fare_imputed|label|prediction|
+------+------+-----------------+-----+-----+------------+-----+----------+
|3     |female|14.0             |0    |0    |7.8542      |0    |1.0       |
|2     |male  |34.0             |0    |0    |13.0        |1    |0.0       |
|3     |female|8.0              |3    |1    |21.075      |0    |1.0       |
|1     |male  |28.0             |1    |0    |82.1708     |0    |1.0       |
|3     |female|18.0             |1    |0    |17.8        |0    |1.0       |
|1     |male  |29.69911764705882|0    |0    |27.7208     |0    |1.0       |
|1     |male  |28.0             |0    |0    |47.1        |0    |1.0       |
|3     |female|28.0             |0    |0    |7.8958      |0    |1.0       |
|1     |male  |24.0             |0    |1    |247.5208    |0    |1.0       |
|3     |male  |12.0             |1    |0    |11.2417     |1    |0.0  

## 12. モデルの係数確認

In [20]:
// Logistic Regressionモデルの係数を表示
val lrModel = model.stages(0).asInstanceOf[org.apache.spark.ml.classification.LogisticRegressionModel]

println("モデルの係数:")
println(s"Intercept: ${lrModel.intercept}")
println(s"Coefficients: ${lrModel.coefficients}")
println()

// 特徴量の重要度（係数の絶対値）
val featureNames = Array(
  "pclass", "age_imputed", "sibsp", "parch", "fare_imputed",
  "sex_vec_0", "embarked_vec_0", "embarked_vec_1", "embarked_vec_2"
)

val coefficients = lrModel.coefficients.toArray

println("特徴量の重要度（係数の絶対値）:")
featureNames.zip(coefficients).sortBy(-_._2.abs).foreach { case (name, coef) =>
  println(f"$name%-20s: $coef%.4f")
}

モデルの係数:
Intercept: 4.988844267577015
Coefficients: [-1.0911480409229986,-0.04175152229991809,-0.3257570425021949,-0.0414704597210331,0.0034608083554357005,-2.578466832548386,-0.07441197802071589,0.10323084582980471,-0.15084245733761184]

特徴量の重要度（係数の絶対値）:
sex_vec_0           : -2.5785
pclass              : -1.0911
sibsp               : -0.3258
embarked_vec_2      : -0.1508
embarked_vec_1      : 0.1032
embarked_vec_0      : -0.0744
age_imputed         : -0.0418
parch               : -0.0415
fare_imputed        : 0.0035


lrModel: org.apache.spark.ml.classification.LogisticRegressionModel = LogisticRegressionModel: uid=logreg_db27a6712be9, numClasses=2, numFeatures=9
featureNames: Array[String] = Array(
  "pclass",
  "age_imputed",
  "sibsp",
  "parch",
  "fare_imputed",
  "sex_vec_0",
  "embarked_vec_0",
  "embarked_vec_1",
  "embarked_vec_2"
)
coefficients: Array[Double] = Array(
  -1.0911480409229986,
  -0.04175152229991809,
  -0.3257570425021949,
  -0.0414704597210331,
  0.0034608083554357005,
  -2.578466832548386,
  -0.07441197802071589,
  0.10323084582980471,
  -0.15084245733761184
)

## 13. 生存予測のシミュレーション

In [21]:
// 性別ごとの生存予測傾向
println("性別ごとの生存予測:")
predictions.groupBy("sex", "prediction").count().orderBy("sex", "prediction").show()

// クラスごとの生存予測傾向
println("\nクラスごとの生存予測:")
predictions.groupBy("pclass", "prediction").count().orderBy("pclass", "prediction").show()

性別ごとの生存予測:
+------+----------+-----+
|   sex|prediction|count|
+------+----------+-----+
|female|       0.0|    5|
|female|       1.0|   72|
|  male|       0.0|  133|
|  male|       1.0|   15|
+------+----------+-----+


クラスごとの生存予測:
+------+----------+-----+
|pclass|prediction|count|
+------+----------+-----+
|     1|       0.0|   18|
|     1|       1.0|   35|
|     2|       0.0|   20|
|     2|       1.0|   15|
|     3|       0.0|  100|
|     3|       1.0|   37|
+------+----------+-----+



## 14. クリーンアップ

In [22]:
// SparkSessionの停止
// spark.stop()
println("完了！")

完了！
